# 00 — Data Inventory

**Blocking notebook.** Before activating Patient-as-Screen, Ghost Karyotypes, forecasting, RNA, or structural branches, verify exactly what the hackathon dataset contains.

This notebook should be run locally against the gated dataset. It inventories metadata only; do not upload sensitive genomic files or raw patient data to third-party services.

## Questions this notebook must answer

- Exact files and formats
- VCF vs BAM/CRAM availability
- Genome build and sample/tissue metadata
- Patient/family relationships
- Bulk RNA-seq?
- scRNA/single-cell?
- Longitudinal samples?
- Clinical karyotype/cytogenetic ground truth?
- Phenotype/HPO data?
- Proteomics, imaging, fibroblast, iPSC or organoid data?
- Access/data-handling restrictions?

In [ ]:
from pathlib import Path
import pandas as pd

# CHANGE THIS to the local folder containing the gated dataset.
DATA_ROOT = Path('../data')

rows = []
if DATA_ROOT.exists():
    for p in sorted(DATA_ROOT.rglob('*')):
        if p.is_file():
            rows.append({
                'relative_path': str(p.relative_to(DATA_ROOT)),
                'suffix': ''.join(p.suffixes).lower(),
                'size_gb': round(p.stat().st_size / (1024**3), 4),
            })
inventory = pd.DataFrame(rows)
inventory

In [ ]:
def detect_modalities(df):
    names = ' '.join(df.get('relative_path', pd.Series(dtype=str)).astype(str)).lower()
    suffixes = set(df.get('suffix', pd.Series(dtype=str)).astype(str))
    return {
        'VCF': any('.vcf' in s for s in suffixes),
        'BAM': any('.bam' in s for s in suffixes),
        'CRAM': any('.cram' in s for s in suffixes),
        'FASTQ': any('.fastq' in s or '.fq' in s for s in suffixes),
        'RNA_HINT': any(k in names for k in ['rna', 'transcript', 'expression']),
        'SINGLE_CELL_HINT': any(k in names for k in ['scrna', 'single_cell', 'single-cell', '10x']),
        'KARYOTYPE_HINT': any(k in names for k in ['karyotype', 'cytogen', 'chromosome']),
        'PHENOTYPE_HINT': any(k in names for k in ['hpo', 'phenotype', 'clinical']),
    }

modalities = detect_modalities(inventory) if not inventory.empty else {}
pd.DataFrame([modalities]).T.rename(columns={0: 'detected_or_filename_hint'})

## Manual verification table

Filename hints are **not proof** of modality. Fill this table from the official dataset manifest/readme and file inspection.

In [ ]:
verification = pd.DataFrame([
    ['VCF / small variants', None, ''],
    ['BAM/CRAM / aligned reads', None, ''],
    ['Bulk RNA-seq', None, ''],
    ['Single-cell / scRNA', None, ''],
    ['Longitudinal timepoints', None, ''],
    ['Clinical karyotype', None, ''],
    ['Phenotype / HPO', None, ''],
    ['Family / parental samples', None, ''],
    ['Proteomics', None, ''],
    ['Imaging', None, ''],
    ['Fibroblast / iPSC / organoid', None, ''],
], columns=['modality', 'confirmed', 'evidence / file'])
verification

In [ ]:
# Update TRUE/FALSE values in `verification` first, then use this cell to decide which branches activate.
confirmed = dict(zip(verification['modality'], verification['confirmed']))
has_reads = bool(confirmed.get('BAM/CRAM / aligned reads'))
has_sc = bool(confirmed.get('Single-cell / scRNA'))
has_long = bool(confirmed.get('Longitudinal timepoints'))
has_vcf = bool(confirmed.get('VCF / small variants'))

branches = pd.DataFrame([
    ['01 Variant discovery', 'ACTIVE' if has_vcf else 'BLOCKED'],
    ['03 Patient-as-Screen / mosaic landscape', 'ACTIVE' if has_sc else ('CONDITIONAL' if has_reads else 'BLOCKED')],
    ['04 Ghost Karyotypes', 'CONDITIONAL' if (has_sc or has_reads) else 'BLOCKED'],
    ['05 Fitness landscape', 'ACTIVE' if (has_sc and has_long) else ('CONDITIONAL' if has_sc else 'BLOCKED')],
    ['08 Mosaic forecast', 'ACTIVE' if (has_sc and has_long) else 'SCENARIO-ONLY / BLOCKED'],
    ['12 Structural mechanism', 'CONDITIONAL ON CAUSAL VARIANT'],
], columns=['branch', 'status'])
branches

## Exit criteria

Do not proceed until the dataset inventory is documented in `research/open_questions.md` (or a dedicated manifest) and the Branch Activation Matrix is reviewed.

**Next notebook after this gate:** `01_variant_discovery.ipynb`.